### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="qsar_aquatic_toxicity",
    dataset_year="2014",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5SG7H",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/505/qsar+aquatic+toxicity.zip && unzip qsar+aquatic+toxicity.zip && rm qsar+aquatic+toxicity.zip && mkdir -p local-data-warehouse/qsar_aquatic_toxicity && mv qsar_aquatic_toxicity.csv local-data-warehouse/qsar_aquatic_toxicity/
""",
    # References
    academic_reference_bibtex="""@article{cassotti2014prediction,
  title={Prediction of acute aquatic toxicity toward daphnia magna by using the ga-k nn method},
  author={Cassotti, Matteo and Ballabio, Davide and Consonni, Viviana and Mauri, Andrea and Tetko, Igor V and Todeschini, Roberto},
  journal={Alternatives to Laboratory Animals},
  volume={42},
  number={1},
  pages={31--41},
  year={2014},
  publisher={SAGE Publications Sage UK: London, England}
}
""",
    academic_reference_bibtex_key="cassotti2014prediction",
    licence="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- Note, the dataset has only a subset of features of the original data. The original data was reduced through feature selection with kNN by the authors in the paper above. We only have the 8 descriptors available in the UCI version. This might leak target information into the feature selection (or behave like an expert giving us the best features). Moreover, it biases the datasets and might make it trivial.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LC50",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
columns = [
    "TPSA",
    "SAacc",
    "H-050",
    "MLOGP",
    "RDCHI",
    "GATS1p",
    "nN",
    "C-040",
    "LC50"
]
df = pd.read_csv(dataset_mold.path / "qsar_aquatic_toxicity.csv", sep=";", names=columns)
print("Loaded data shape:", df.shape)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (546, 9)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 546
Columns: 9
Use sampling: False (sample size: 546)
Get row duplicates (staged, merged)...
Using top-8 columns for initial filtering: ['MLOGP', 'GATS1p', 'RDCHI', 'TPSA', 'SAacc', 'H-050', 'nN', 'C-040']
Rows remaining as candidates after top-8 filter: 16 (of 546)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 9 (1.65% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,TPSA,SAacc,H-050,MLOGP,RDCHI,GATS1p,nN,C-040,LC50
0,90.37,131.635,3,0.332,2.706,1.313,3,1,2.072
1,38.33,54.156,1,1.737,2.472,0.638,1,0,6.848
2,84.58,129.736,4,0.925,3.178,1.263,2,1,3.902
3,38.80,0.000,1,1.364,1.334,1.212,0,0,6.102
4,20.23,42.683,1,2.193,1.960,1.020,0,0,4.038


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,TPSA,float64,0.0,0.0,227.0,"0.0, 20.23, 26.02, 45.82, 9.23, 17.07, 29.46, 52.6, 37.3, 40.46"
1,SAacc,float64,0.0,0.0,210.0,"0.0, 42.683, 11.0, 32.897, 50.747, 16.786, 53.683, 25.145, 85.367, 28.269"
2,MLOGP,float64,0.0,0.0,405.0,"1.859, 2.226, 2.193, 1.246, 2.729, 2.127, 3.314, -0.317, 3.11, -0.273"
3,RDCHI,float64,0.0,0.0,342.0,"1.334, 1.975, 1.401, 1.509, 2.031, 1.155, 1.225, 1.924, 1.918, 1.908"
4,GATS1p,float64,0.0,0.0,403.0,"0.478, 1.081, 1.15, 0.867, 0.462, 0.942, 0.575, 1.111, 0.917, 1.063"
5,LC50,float64,0.0,0.0,515.0,"3.85, 1.22, 3.339, 3.277, 3.432, 3.884, 5.6, 4.34, 6.064, 5.551"
6,H-050,int64,0.0,0.0,11.0,"0, 1, 2, 3, 4, 5, 6, 7, 8, 16"
7,nN,int64,0.0,0.0,9.0,"0, 1, 2, 3, 5, 4, 7, 6, 11"
8,C-040,int64,0.0,0.0,6.0,"0, 1, 2, 4, 3, 11"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
TPSA,546.0,48.472930,46.763983,0.000,347.320
SAacc,546.0,58.869018,68.166554,0.000,571.952
H-050,546.0,0.937729,1.618632,0.000,18.000
MLOGP,546.0,2.313493,1.741797,-6.446,9.148
RDCHI,546.0,2.492299,0.811004,1.000,6.439
GATS1p,546.0,1.046264,0.403677,0.281,2.500
nN,546.0,1.003663,1.397240,0.000,11.000
C-040,546.0,0.353480,0.806827,0.000,11.000
LC50,546.0,4.658421,1.665215,0.122,10.047


In [7]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.321,-2.254,2.773,0.21,log,2774.6,175686.2,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7bba-c06d-7864-9f69-725801a1ace7
073cb1fbc9530840444670aa04b7b92f3b5362d114dfe9f199ccb4c630989f65
